In [7]:
!pip install pandas numpy gdown stanza

  Using cached stanza-1.11.1-py3-none-any.whl.metadata (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.2/337.2 kB 21.0 MB/s eta 0:00:00


In [ ]:
import stanza
stanza.download("uk")

nlp = stanza.Pipeline(
    lang="uk",
    processors="tokenize,pos,lemma",
    tokenize_pretokenized=True,
    use_gpu=False
)

INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: uk (Ukrainian) ...
INFO:stanza:File exists: /root/.cache/stanza/1.11.0/resources/uk/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Loading these models for language: uk (Ukrainian):
| Processor | Package     |
---------------------------
| tokenize  | iu          |
| pos       | iu_charlm   |
| lemma     | iu_nocharlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Done loading processors!


In [ ]:
def process_from_sentences(text_dict):
    sentences = text_dict["sentences"]

    tokenized_input = [s.split() for s in sentences]

    doc = nlp(tokenized_input)

    lemmas = []
    upos_tags = []

    for sentence in doc.sentences:
        for word in sentence.words:
            lemmas.append(word.lemma)
            upos_tags.append(word.upos)

    return " ".join(lemmas), " ".join(upos_tags)

In [ ]:
import pandas as pd
import gdown

file_id = "19hOT_T7Jr82N6u9XXSBEcM7P2GI-6rWL"

url = f"https://drive.google.com/uc?id={file_id}"

output = "processed_v2.csv"
gdown.download(url, output, quiet=False)

df = pd.read_csv(output)
df.head(5)


Downloading...
From: https://drive.google.com/uc?id=19hOT_T7Jr82N6u9XXSBEcM7P2GI-6rWL
To: /content/processed_v2.csv
100%|██████████| 4.40M/4.40M [00:00<00:00, 33.8MB/s]


,title,text,category_id
0,Івано-Франківський драмтеатр знайшов архівні д...,{'clean': 'Про це на пресконференції повідомив...,0
1,Премія Олеся Гончара оголосила номінантів,"{'clean': 'Як передає Укрінформ, про це повідо...",0
2,Біографічний фільм «Я граю Роккі» вийде у лист...,"{'clean': 'Як передає Укрінформ, про це повідо...",0
3,Netflix підписав угоду на трансляцію фільмів S...,"{'clean': 'Про це повідомила Deadline, передає...",0
4,На Венеційській бієнале Україну представить ск...,{'clean': 'Про це розповіли учасники Венеційсь...,0


In [ ]:
import ast
df["text"] = df["text"].apply(ast.literal_eval)


df[["lemma_text", "upos_seq"]] = df["text"].apply(
    lambda x: pd.Series(process_from_sentences(x))
)

df.to_csv("processed_v2_lemma.csv", index=False)

In [19]:
import gdown
import pandas as pd
file_id = "1ypHTyWvPif9MRipCdqlxELAup6hrvfYh"

url = f"https://drive.google.com/uc?id={file_id}"

output = "processed_v2_lemma.csv"
gdown.download(url, output, quiet=False)

df = pd.read_csv(output)
df.head(5)


Downloading...
From: https://drive.google.com/uc?id=1ypHTyWvPif9MRipCdqlxELAup6hrvfYh
To: /content/processed_v2_lemma.csv
100%|██████████| 7.28M/7.28M [00:00<00:00, 23.9MB/s]


,title,text,category_id,lemma_text,upos_seq
0,Івано-Франківський драмтеатр знайшов архівні д...,{'clean': 'Про це на пресконференції повідомив...,0,про це на пресконфенція повідомити гендиректор...,ADP PRON ADP NOUN VERB ADJ NOUN ADJ NOUN ADJ N...
1,Премія Олеся Гончара оголосила номінантів,"{'clean': 'Як передає Укрінформ, про це повідо...",0,"як передавати Укрінформ, про це повідомити Дер...",SCONJ VERB PROPN ADP PRON VERB PROPN ADV ADP N...
2,Біографічний фільм «Я граю Роккі» вийде у лист...,"{'clean': 'Як передає Укрінформ, про це повідо...",0,"як передавати Укрінформ, про це повідомляти Th...",SCONJ VERB PROPN ADP PRON VERB X X X NOUN ADP ...
3,Netflix підписав угоду на трансляцію фільмів S...,"{'clean': 'Про це повідомила Deadline, передає...",0,"про це повідомити Deadline, передавати Укрінфо...",ADP PRON VERB X VERB PROPN VERB SCONJ ADJ NOUN...
4,На Венеційській бієнале Україну представить ск...,{'clean': 'Про це розповіли учасники Венеційсь...,0,про це розповісти учасник венеційський бієнале...,ADP PRON VERB NOUN ADJ NOUN ADP PROPN ADP NOUN...


In [20]:
from sklearn.model_selection import train_test_split
import ast

X_proc = df["text"].apply(ast.literal_eval).apply(lambda x: x["clean"])
X_lemma = df["lemma_text"]
y = df["category_id"]

X_train_proc, X_test_proc, y_train, y_test = train_test_split(
    X_proc, y, test_size=0.2, random_state=42, stratify=y
)

X_train_lemma, X_test_lemma, _, _ = train_test_split(
    X_lemma, y, test_size=0.2, random_state=42, stratify=y
)

In [21]:
from sklearn.metrics import confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score


def run_experiment(X_train, X_test, y_train, y_test, model):
    vectorizer = TfidfVectorizer(
        ngram_range=(1,2),
        max_features=30000,
        min_df=3
    )

    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)

    cm = confusion_matrix(y_test, y_pred)
    print("Confusion matrix:")
    print(cm)

    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")

    return acc, macro_f1

In [22]:
logreg = LogisticRegression(
    max_iter=1000,
    n_jobs=-1
)

acc_proc_lr, f1_proc_lr = run_experiment(
    X_train_proc, X_test_proc, y_train, y_test, logreg
)

acc_lemma_lr, f1_lemma_lr = run_experiment(
    X_train_lemma, X_test_lemma, y_train, y_test, logreg
)

Confusion matrix:
[[42  0  0  0]
 [ 0 42  0  0]
 [ 0  0 39  3]
 [ 1  1  2 38]]
Confusion matrix:
[[41  0  1  0]
 [ 0 42  0  0]
 [ 0  0 39  3]
 [ 0  1  3 38]]


In [23]:
svm = LinearSVC(
    C=1.0,
    max_iter=2000
)

acc_proc_svm, f1_proc_svm = run_experiment(
    X_train_proc, X_test_proc, y_train, y_test, svm
)

acc_lemma_svm, f1_lemma_svm = run_experiment(
    X_train_lemma, X_test_lemma, y_train, y_test, svm
)

Confusion matrix:
[[42  0  0  0]
 [ 0 42  0  0]
 [ 0  0 39  3]
 [ 1  1  2 38]]
Confusion matrix:
[[42  0  0  0]
 [ 0 42  0  0]
 [ 0  0 39  3]
 [ 0  1  3 38]]


In [24]:
print("=== Logistic Regression ===")
print("Processed_v2:")
print("Accuracy:", acc_proc_lr)
print("Macro-F1:", f1_proc_lr)

print("\nLemma_text:")
print("Accuracy:", acc_lemma_lr)
print("Macro-F1:", f1_lemma_lr)


print("\n=== Linear SVM ===")
print("Processed_v2:")
print("Accuracy:", acc_proc_svm)
print("Macro-F1:", f1_proc_svm)

print("\nLemma_text:")
print("Accuracy:", acc_lemma_svm)
print("Macro-F1:", f1_lemma_svm)

=== Logistic Regression ===
Processed_v2:
Accuracy: 0.9583333333333334
Macro-F1: 0.9579730687455705

Lemma_text:
Accuracy: 0.9523809523809523
Macro-F1: 0.9523742026931253

=== Linear SVM ===
Processed_v2:
Accuracy: 0.9583333333333334
Macro-F1: 0.9579730687455705

Lemma_text:
Accuracy: 0.9583333333333334
Macro-F1: 0.9581173433228714


In [ ]:
report = f"""
# Lab 3 — NLP Audit & Baseline Comparison

## Baseline: TF-IDF + Logistic Regression

| Text version | Accuracy | Macro-F1 |
|--------------|----------|----------|
| processed_v2 | {acc_proc_lr:.4f} | {f1_proc_lr:.4f} |
| lemma_text   | {acc_lemma_lr:.4f} | {f1_lemma_lr:.4f} |

---

## Baseline: TF-IDF + Linear SVM

| Text version | Accuracy | Macro-F1 |
|--------------|----------|----------|
| processed_v2 | {acc_proc_svm:.4f} | {f1_proc_svm:.4f} |
| lemma_text   | {acc_lemma_svm:.4f} | {f1_lemma_svm:.4f} |

---

## Порівняння

### Logistic Regression
- Різниця Accuracy: {(acc_proc_lr - acc_lemma_lr):.4f}
- Різниця Macro-F1: {(f1_proc_lr - f1_lemma_lr):.4f}

### Linear SVM
- Різниця Accuracy: {(acc_proc_svm - acc_lemma_svm):.4f}
- Різниця Macro-F1: {(f1_proc_svm - f1_lemma_svm):.4f}

---

## Аналіз критичних помилок

### 1. Втрата ваги ключових сутностей (Paramount / Netflix / Warner Bros.)
У лематизованій версії ключові назви компаній розмічені як `X` замість `PROPN`, а складна конструкція "кіно- та телестудіями" узагальнюється.
**Наслідок:** модель втрачає сильні маркери індустрії розваг і може зміщувати текст у бік економіки або бізнесу.

### 2. Руйнування власних назв франшиз
Назви:
- "Гаррі Поттер"
- "Гра престолів"
- "Друзі"
- "DC Comics"
пошкоджені або частково спотворені ("престоліве", "ррузи").
**Наслідок:** культурні маркери тематики зникають або перетворюються на noise-токени.

### 3. Зсув у бік економічної лексики
У тексті з великими сумами ("82,7 мільярда доларів", "108,4 мільярда доларів") лематизація підвищує вагу слів:
- "мільярд"
- "долар"
При цьому втрачається контекст злиття медіакомпаній.
**Наслідок:** текст може класифікуватися як економічна новина замість медіа/культури.

### 4. Морфологічні спотворення
Некоректні леми "доларіва", "повідомлялоситися" створюють неіснуючі токени.
**Наслідок:** збільшення шуму в словнику TF-IDF.

### 5. Руйнування сталих виразів
Лематизація перетворює "Білого дому" на "білий дім", не зберігаючи статус власної назви.
**Наслідок:** політичні сутності втрачають специфічність.

---

## Загальний висновок

1. Лематизація зменшує морфологічну варіативність, але може руйнувати власні назви та генерувати шумові токени.
2. processed_v2 краще зберігає більше контексту: бренди, франшизи та складні власні назви.
3. Лематизована версія тексту lemma_text не продемонструвала покращення метрик (Accuracy та Macro-F1) порівняно з processed_v2.
4. У межах даного корпусу обидва baseline-підходи (Logistic Regression та Linear SVM) продемонстрували майже однакову ефективність, однак стабільно кращі результати були отримані для версії processed_v2.
5. Для заданого корпусу та задачі новинної класифікації використання сирого processed_v2 ефективніше, ніж повна лематизація.

"""

with open("audit_summary_lab3.md", "w", encoding="utf-8") as f:
    f.write(report)

